In [1]:
#!pip install yfinance pandas-datareader
import pandas as pd
import yfinance as yf
import pandas_datareader.data as web
import datetime
import warnings
warnings.filterwarnings("ignore")
print("Import thư viện thành công!")

Import thư viện thành công!


# PHẦN 1. XÂY DỰNG DỮ LIỆU VÀ CƠ SỞ LÝ THUYẾT
## Phân tích tác động của lãi suất và lạm phát đến lợi suất cổ phiếu Microsoft (MSFT)
**Cổ phiếu:** Microsoft Corporation (MSFT)  
**Thời gian nghiên cứu:** 01/2021 – 08/2026  
**Tần suất dữ liệu:** Monthly  
### Các biến nghiên cứu
- **Y:** Monthly Return của MSFT
- **X1:** Federal Funds Effective Rate (Interest Rate)
- **X2:** Inflation Rate (CPI YoY)
### Nguồn dữ liệu
- Yahoo Finance: dữ liệu giá cổ phiếu MSFT
- FRED: Federal Funds Effective Rate (FEDFUNDS)
- FRED: Consumer Price Index (CPIAUCSL)

## 1.2. Thu thập dữ liệu
Dữ liệu được thu thập theo tần suất tháng (Monthly) trong khoảng thời gian từ tháng 01/2021 đến tháng 08/2026.
Đối với dữ liệu cổ phiếu Microsoft, nghiên cứu sử dụng giá đóng cửa điều chỉnh (Adjusted Close) từ Yahoo Finance. Adjusted Close được sử dụng làm giá đại diện để tính Monthly Return.
Đối với lãi suất, nghiên cứu sử dụng Federal Funds Effective Rate (FEDFUNDS) từ FRED.
Đối với lạm phát, nghiên cứu sử dụng chỉ số CPIAUCSL từ FRED và tính tỷ lệ lạm phát theo phương pháp Year-over-Year (YoY).

In [2]:
# Lấy dữ liệu từ năm 2020 để có đủ dữ liệu CPI
# phục vụ tính Inflation YoY cho các tháng đầu năm 2021.
start_date = datetime.datetime(2020, 1, 1)
end_date = datetime.datetime(2026, 9, 1)
print("Start date:", start_date.date())
print("End date:", end_date.date())

Start date: 2020-01-01
End date: 2026-09-01


In [3]:
print("Đang tải dữ liệu MSFT từ Yahoo Finance...")
msft_df = yf.download(
    "MSFT",
    start=start_date,
    end=end_date,
    auto_adjust=False,
    progress=False
)
print("Đã tải dữ liệu MSFT.")
print("Số dòng:", len(msft_df))
display(msft_df.head())

Đang tải dữ liệu MSFT từ Yahoo Finance...
Đã tải dữ liệu MSFT.
Số dòng: 1674


Price,Adj Close,Close,High,Low,Open,Volume
Ticker,MSFT,MSFT,MSFT,MSFT,MSFT,MSFT
Date,,,,,,
2020-01-02,151.544281,160.619995,160.729996,158.330002,158.779999,22622100
2020-01-03,149.657227,158.619995,159.949997,158.059998,158.320007,21116200
2020-01-06,150.044098,159.029999,159.100006,156.509995,157.080002,20813700
2020-01-07,148.676041,157.580002,159.669998,157.320007,159.320007,21634100
2020-01-08,151.044220,160.089996,160.800003,157.949997,158.929993,27746500


In [4]:
print("Các cột dữ liệu:")
print(msft_df.columns)
print("\nThông tin dữ liệu:")
msft_df.info()

Các cột dữ liệu:
MultiIndex([('Adj Close', 'MSFT'),
            (    'Close', 'MSFT'),
            (     'High', 'MSFT'),
            (      'Low', 'MSFT'),
            (     'Open', 'MSFT'),
            (   'Volume', 'MSFT')],
           names=['Price', 'Ticker'])

Thông tin dữ liệu:
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 1674 entries, 2020-01-02 to 2026-08-31
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   (Adj Close, MSFT)  1674 non-null   float64
 1   (Close, MSFT)      1674 non-null   float64
 2   (High, MSFT)       1674 non-null   float64
 3   (Low, MSFT)        1674 non-null   float64
 4   (Open, MSFT)       1674 non-null   float64
 5   (Volume, MSFT)     1674 non-null   int64  
dtypes: float64(5), int64(1)
memory usage: 91.5 KB


In [5]:
# Lấy Adjusted Close
msft_monthly = (
    msft_df["Adj Close"]
    .resample("ME")
    .last()
)
if isinstance(msft_monthly, pd.DataFrame):
    msft_monthly = msft_monthly["MSFT"]
msft_monthly = msft_monthly.to_frame(name="MSFT_Price")
print("Dữ liệu MSFT theo tháng:")
display(msft_monthly.head())
print("\n5 tháng cuối:")
display(msft_monthly.tail())

Dữ liệu MSFT theo tháng:


,MSFT_Price
Date,
2020-01-31,160.611267
2020-02-29,153.273224
2020-03-31,149.205109
2020-04-30,169.545670
2020-05-31,173.850677



5 tháng cuối:


,MSFT_Price
Date,
2026-04-30,406.134155
2026-05-31,449.394012
2026-06-30,372.319092
2026-07-31,463.846802
2026-08-31,507.290009


### Giải thích
Dữ liệu giá cổ phiếu ban đầu được cung cấp theo ngày giao dịch. Để thống nhất với dữ liệu kinh tế vĩ mô, giá cổ phiếu được chuyển sang tần suất tháng.
Giá đại diện cho mỗi tháng là Adjusted Close của ngày giao dịch cuối cùng trong tháng.
Biến `MSFT_Price` chưa phải là biến phụ thuộc cuối cùng của mô hình. Biến này sẽ được sử dụng để tính Monthly Return.

In [6]:
print("Đang tải dữ liệu vĩ mô từ FRED...")
macro_df = web.DataReader(
    ["FEDFUNDS", "CPIAUCSL"],
    "fred",
    start_date,
    end_date
)
macro_df.columns = [
    "Interest_Rate",
    "CPI"
]
print("Đã tải dữ liệu FRED.")
display(macro_df.head())

Đang tải dữ liệu vĩ mô từ FRED...
Đã tải dữ liệu FRED.


,Interest_Rate,CPI
DATE,,
2020-01-01,1.55,259.127
2020-02-01,1.58,259.250
2020-03-01,0.65,258.076
2020-04-01,0.05,256.032
2020-05-01,0.05,255.802


In [7]:
# Chuyển ngày của dữ liệu FRED về cuối tháng
macro_df.index = (
    macro_df.index
    .to_period("M")
    .to_timestamp("M")
)
macro_df.index.name = "Date"
print("Dữ liệu FRED sau khi chuẩn hóa thời gian:")
display(macro_df.head())

Dữ liệu FRED sau khi chuẩn hóa thời gian:


,Interest_Rate,CPI
Date,,
2020-01-31,1.55,259.127
2020-02-29,1.58,259.250
2020-03-31,0.65,258.076
2020-04-30,0.05,256.032
2020-05-31,0.05,255.802


## 1.3. Mô tả các biến nghiên cứu
### Biến phụ thuộc – Monthly Return
Biến phụ thuộc của mô hình là Monthly Return của cổ phiếu MSFT.
Monthly Return được tính dựa trên Adjusted Close cuối mỗi tháng:
Monthly Return_t = ((P_t - P_(t-1)) / P_(t-1)) × 100
Trong đó:
- P_t là Adjusted Close của MSFT tại tháng t.
- P_(t-1) là Adjusted Close của MSFT tại tháng trước.
### Biến độc lập X1 – Interest Rate
Interest Rate được đại diện bởi Federal Funds Effective Rate (FEDFUNDS), đơn vị %.
### Biến độc lập X2 – Inflation Rate
Inflation Rate được tính từ CPI theo phương pháp Year-over-Year:
Inflation_t = ((CPI_t / CPI_(t-12)) - 1) × 100
CPI_t là CPI của tháng hiện tại và CPI_(t-12) là CPI của cùng tháng năm trước.

In [8]:
# Tính tỷ lệ lạm phát YoY
macro_df["Inflation_Rate"] = (
    macro_df["CPI"].pct_change(12) * 100
)
print("Dữ liệu sau khi tính Inflation Rate:")
display(
    macro_df[
        ["CPI", "Inflation_Rate"]
    ].head(15)
)

Dữ liệu sau khi tính Inflation Rate:


,CPI,Inflation_Rate
Date,,
2020-01-31,259.127,NaN
2020-02-29,259.250,NaN
2020-03-31,258.076,NaN
2020-04-30,256.032,NaN
2020-05-31,255.802,NaN
2020-06-30,257.042,NaN
2020-07-31,258.352,NaN
2020-08-31,259.316,NaN
2020-09-30,259.997,NaN


In [9]:
print("Một số giá trị Inflation Rate:")
display(
    macro_df[
        ["CPI", "Inflation_Rate"]
    ].dropna().head(10)
)
print("\n5 tháng gần nhất:")
display(
    macro_df[
        ["CPI", "Inflation_Rate"]
    ].tail()
)

Một số giá trị Inflation Rate:


,CPI,Inflation_Rate
Date,,
2021-01-31,262.687,1.373844
2021-02-28,263.579,1.669817
2021-03-31,264.961,2.667819
2021-04-30,266.614,4.133077
2021-05-31,268.383,4.918257
2021-06-30,270.654,5.295633
2021-07-31,271.903,5.245169
2021-08-31,272.676,5.152015
2021-09-30,273.910,5.351216



5 tháng gần nhất:


,CPI,Inflation_Rate
Date,,
2026-04-30,332.407,3.779246
2026-05-31,333.979,4.166615
2026-06-30,332.568,3.463531
2026-07-31,332.813,3.303856
2026-08-31,334.131,3.353016


In [10]:
# Ghép dữ liệu MSFT với dữ liệu vĩ mô
df = pd.merge(
    msft_monthly,
    macro_df[
        ["Interest_Rate", "Inflation_Rate"]
    ],
    left_index=True,
    right_index=True,
    how="inner"
)
# Chỉ lấy giai đoạn nghiên cứu
df = df.loc[
    "2021-01-01":"2026-08-31"
].copy()
df.index.name = "Date"
print("Dữ liệu sau khi ghép:")
display(df.head())
print("\n5 dòng cuối:")
display(df.tail())

Dữ liệu sau khi ghép:


,MSFT_Price,Interest_Rate,Inflation_Rate
Date,,,
2021-01-31,221.171677,0.09,1.373844
2021-02-28,222.082474,0.08,1.669817
2021-03-31,225.322235,0.07,2.667819
2021-04-30,241.005051,0.07,4.133077
2021-05-31,239.166809,0.06,4.918257



5 dòng cuối:


,MSFT_Price,Interest_Rate,Inflation_Rate
Date,,,
2026-04-30,406.134155,3.64,3.779246
2026-05-31,449.394012,3.63,4.166615
2026-06-30,372.319092,3.63,3.463531
2026-07-31,463.846802,3.63,3.303856
2026-08-31,507.290009,3.63,3.353016


In [11]:
print("Kích thước dữ liệu:", df.shape)
print("\nTên các cột:")
print(df.columns.tolist())
print("\nKhoảng thời gian:")
print("Từ:", df.index.min())
print("Đến:", df.index.max())

Kích thước dữ liệu: (68, 3)

Tên các cột:
['MSFT_Price', 'Interest_Rate', 'Inflation_Rate']

Khoảng thời gian:
Từ: 2021-01-31 00:00:00
Đến: 2026-08-31 00:00:00


## 1.4. Tiền xử lý dữ liệu
Sau khi ghép dữ liệu, tiến hành kiểm tra các giá trị bị thiếu, dữ liệu trùng lặp và kiểu dữ liệu.
Dữ liệu được lọc trong khoảng thời gian nghiên cứu từ tháng 01/2021 đến tháng 08/2026.
Các giá trị thiếu được kiểm tra trước khi xử lý. Nếu không có giá trị thiếu thì giữ nguyên dữ liệu. Nếu xuất hiện giá trị thiếu, các quan sát không đầy đủ sẽ được loại bỏ để tránh đưa dữ liệu không xác định vào mô hình.

In [12]:
print("=== KIỂM TRA MISSING VALUES ===")
missing_values = df.isnull().sum()
display(missing_values)

=== KIỂM TRA MISSING VALUES ===


MSFT_Price        0
Interest_Rate     0
Inflation_Rate    0
dtype: int64

In [13]:
print("=== KIỂM TRA DỮ LIỆU TRÙNG LẶP ===")
duplicate_dates = df.index.duplicated().sum()
print("Số ngày bị trùng:", duplicate_dates)

=== KIỂM TRA DỮ LIỆU TRÙNG LẶP ===
Số ngày bị trùng: 0


In [14]:
# Nếu có missing values thì loại bỏ các dòng bị thiếu
df = df.dropna().copy()
print("Missing values sau khi xử lý:")
print(df.isnull().sum())

Missing values sau khi xử lý:
MSFT_Price        0
Interest_Rate     0
Inflation_Rate    0
dtype: int64


In [15]:
print("=== THÔNG TIN DATAFRAME ===")
df.info()

=== THÔNG TIN DATAFRAME ===
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 68 entries, 2021-01-31 to 2026-08-31
Freq: ME
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   MSFT_Price      68 non-null     float64
 1   Interest_Rate   68 non-null     float64
 2   Inflation_Rate  68 non-null     float64
dtypes: float64(3)
memory usage: 2.1 KB


In [16]:
# Làm tròn 4 chữ số thập phân để dữ liệu dễ quan sát
df = df.round(4)
display(df.head(10))

,MSFT_Price,Interest_Rate,Inflation_Rate
Date,,,
2021-01-31,221.1717,0.09,1.3738
2021-02-28,222.0825,0.08,1.6698
2021-03-31,225.3222,0.07,2.6678
2021-04-30,241.0051,0.07,4.1331
2021-05-31,239.1668,0.06,4.9183
2021-06-30,259.4933,0.08,5.2956
2021-07-31,272.9135,0.10,5.2452
2021-08-31,289.7225,0.09,5.1520
2021-09-30,270.5663,0.08,5.3512


In [17]:
print("=== THỐNG KÊ CƠ BẢN ===")
display(df.describe())

=== THỐNG KÊ CƠ BẢN ===


,MSFT_Price,Interest_Rate,Inflation_Rate
count,68.000000,68.000000,68.000000
mean,351.464890,3.276471,4.350325
std,88.887566,2.005910,2.081751
min,221.171700,0.060000,1.373800
25%,270.977750,1.100000,2.786850
50%,350.111700,4.095000,3.334450
75%,411.499325,4.887500,5.502850
max,528.278000,5.330000,8.979400


In [18]:
print("=== DATASET CUỐI CÙNG ===")
print("Số dòng:", df.shape[0])
print("Số cột:", df.shape[1])
print("\nKhoảng thời gian:")
print(df.index.min(), "đến", df.index.max())
print("\nCác cột:")
print(df.columns.tolist())
display(df.head())
display(df.tail())

=== DATASET CUỐI CÙNG ===
Số dòng: 68
Số cột: 3

Khoảng thời gian:
2021-01-31 00:00:00 đến 2026-08-31 00:00:00

Các cột:
['MSFT_Price', 'Interest_Rate', 'Inflation_Rate']


,MSFT_Price,Interest_Rate,Inflation_Rate
Date,,,
2021-01-31,221.1717,0.09,1.3738
2021-02-28,222.0825,0.08,1.6698
2021-03-31,225.3222,0.07,2.6678
2021-04-30,241.0051,0.07,4.1331
2021-05-31,239.1668,0.06,4.9183


,MSFT_Price,Interest_Rate,Inflation_Rate
Date,,,
2026-04-30,406.1342,3.64,3.7792
2026-05-31,449.3940,3.63,4.1666
2026-06-30,372.3191,3.63,3.4635
2026-07-31,463.8468,3.63,3.3039
2026-08-31,507.2900,3.63,3.3530


In [19]:
# Lưu dataset vào thư mục data
output_path = "../data/msft_macro_2021_2026.csv"
df.to_csv(output_path)
print("Đã lưu dữ liệu tại:")
print(output_path)

Đã lưu dữ liệu tại:
../data/msft_macro_2021_2026.csv


## 1.5. Cơ sở lý thuyết
### 1.5.1. Tác động của lãi suất
Lãi suất có thể ảnh hưởng đến giá cổ phiếu thông qua tỷ lệ chiết khấu và chi phí vốn. Khi lãi suất tăng, giá trị hiện tại của các dòng tiền kỳ vọng trong tương lai có thể giảm. Bên cạnh đó, chi phí vốn tăng có thể ảnh hưởng đến hoạt động đầu tư và chi tiêu của doanh nghiệp.
Đối với các công ty công nghệ như Microsoft, sự thay đổi của lãi suất có thể ảnh hưởng đến kỳ vọng của nhà đầu tư đối với tăng trưởng và lợi nhuận trong tương lai. Tuy nhiên, chiều hướng và mức độ tác động thực tế cần được kiểm định bằng dữ liệu.
### 1.5.2. Tác động của lạm phát
Lạm phát có thể làm tăng một số chi phí hoạt động của doanh nghiệp như chi phí nhân sự, năng lượng và cơ sở hạ tầng. Nếu chi phí tăng nhanh hơn doanh thu, lợi nhuận của doanh nghiệp có thể chịu ảnh hưởng.
Ngoài ra, lạm phát còn có thể tác động gián tiếp đến thị trường chứng khoán thông qua chính sách tiền tệ và lãi suất. Vì vậy, lạm phát có thể có mối quan hệ với lợi suất cổ phiếu Microsoft và cần được kiểm định bằng mô hình hồi quy.
### 1.5.3. Mô hình nghiên cứu
Nghiên cứu sử dụng mô hình hồi quy tuyến tính đa biến:
**Monthly Return_t = β0 + β1 Interest Rate_t + β2 Inflation Rate_t + ε_t**
Trong đó:
- **Monthly Return:** lợi suất cổ phiếu MSFT theo tháng.
- **Interest Rate:** Federal Funds Effective Rate.
- **Inflation Rate:** tỷ lệ lạm phát CPI YoY.
- **β0:** hệ số chặn.
- **β1, β2:** hệ số hồi quy.
- **ε:** sai số của mô hình.
Nghiên cứu sẽ sử dụng kết quả hồi quy để đánh giá chiều hướng, mức độ và ý nghĩa thống kê của mối quan hệ giữa các biến.

## Kết luận phần 1
Dữ liệu MSFT, lãi suất và lạm phát đã được thu thập, chuẩn hóa và kiểm tra trong giai đoạn 01/2021–08/2026.
Bộ dữ liệu sau xử lý được lưu thành file CSV và sẽ được sử dụng cho các bước phân tích tiếp theo, bao gồm tính Monthly Return, phân tích dữ liệu và xây dựng mô hình hồi quy tuyến tính đa biến.